# Historical Features (`pipelines/03_historical_features.ipynb`)

**Reads:** `workspace.sentinel_silver.transactions`
**Writes:** `workspace.sentinel_gold.transaction_features` (one row per transaction)

**Flow:** `read Silver → join earlier same-card payments → aggregate 1h & 14d features → save Gold feature table`

For each transaction we look back at the **earlier** payments of the same card and count them. These numbers are what the monitoring rules (notebook 04) use.

| Feature | Meaning |
| --- | --- |
| `card_txn_count_1h` | Earlier payments of this card in the last hour |
| `card_txn_count_14d` | Earlier payments of this card in the last 14 days |
| `card_avg_amount_14d` | Average amount of those 14-day payments (`NULL` when there are none) |
| `card_merchant_txn_count_1h` | Earlier payments of this card at the same merchant in the last hour |
| `card_merchant_similar_amount_count_1h` | Same as above, with an amount within ±10% of the current one |

**"Earlier" rule:** a payment counts if `event_time - window <= its event_time < event_time`.
The current payment, payments at the exact same time, and future payments are never counted. This keeps future activity out of the features.

In [ ]:
from datetime import datetime
from decimal import Decimal
from pyspark.sql import functions as F

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
GOLD_SCHEMA = "workspace.sentinel_gold"
FEATURES_TABLE = f"{GOLD_SCHEMA}.transaction_features"

ONE_HOUR = F.expr("INTERVAL 1 HOUR")
FOURTEEN_DAYS = F.expr("INTERVAL 14 DAYS")

# "Similar amount" means within ±10% of the current amount.
SIMILAR_AMOUNT_TOLERANCE = Decimal("0.10")
LOWER_FACTOR = Decimal("1.00") - SIMILAR_AMOUNT_TOLERANCE  # 0.90
UPPER_FACTOR = Decimal("1.00") + SIMILAR_AMOUNT_TOLERANCE  # 1.10

## How the features are calculated

1. Make a copy of the transactions with renamed columns (`prev_*`). This is the "earlier payments" table.
2. **Left join** each transaction to its earlier payments: same card, strictly earlier time (`prev_event_time < event_time`), and not more than 14 days before (`prev_event_time >= event_time - INTERVAL 14 DAYS`). A transaction with no history keeps one row with `NULL` `prev_*` values.
3. `groupBy("trans_num")` and aggregate. `F.count(F.when(condition, 1))` counts only the rows where the condition is true. The 1-hour counts are just stricter conditions on the same joined rows.
4. Join the counts back to the original transaction columns.

**Small example:** card A pays at 10:00, 10:20, and 10:40. For the 10:40 payment the earlier payments are 10:00 and 10:20, so `card_txn_count_1h = 2`. For the 10:00 payment there are none, so `card_txn_count_14d = 0` and `card_avg_amount_14d = NULL`.

In [ ]:
def add_history_features(transactions):
    # Bounds are not rounded here, so the comparison with prior amounts is exact.
    current = (
        transactions
        .withColumn("similar_amount_lower_bound", F.col("amount") * F.lit(LOWER_FACTOR))
        .withColumn("similar_amount_upper_bound", F.col("amount") * F.lit(UPPER_FACTOR))
    )

    earlier = transactions.select(
        F.col("trans_num").alias("prev_trans_num"),
        F.col("cc_num").alias("prev_cc_num"),
        F.col("merchant").alias("prev_merchant"),
        F.col("amount").alias("prev_amount"),
        F.col("event_time").alias("prev_event_time"),
    )

    # Same card, strictly earlier, at most 14 days before.
    is_earlier_payment = (
        (F.col("cc_num") == F.col("prev_cc_num"))
        & (F.col("prev_event_time") < F.col("event_time"))
        & (F.col("prev_event_time") >= F.col("event_time") - FOURTEEN_DAYS)
    )
    history = current.join(earlier, is_earlier_payment, "left")

    # The 1-hour features only look at the part of the history inside the last hour.
    in_last_hour = F.col("prev_event_time") >= F.col("event_time") - ONE_HOUR
    same_merchant = F.col("prev_merchant") == F.col("merchant")
    similar_amount = F.col("prev_amount").between(
        F.col("similar_amount_lower_bound"), F.col("similar_amount_upper_bound")
    )

    counts = history.groupBy("trans_num").agg(
        F.count("prev_trans_num").alias("card_txn_count_14d"),
        F.round(F.avg("prev_amount"), 2).cast("decimal(18,2)").alias("card_avg_amount_14d"),
        F.count(F.when(in_last_hour, 1)).alias("card_txn_count_1h"),
        F.count(F.when(in_last_hour & same_merchant, 1)).alias("card_merchant_txn_count_1h"),
        F.count(F.when(in_last_hour & same_merchant & similar_amount, 1)).alias(
            "card_merchant_similar_amount_count_1h"
        ),
    )

    return current.join(counts, on="trans_num", how="left")

## Self-test on hand-made payments

The time of the payment being checked is **T0 = 2019-01-15 12:00:00**.

| Payment | When | What it proves |
| --- | --- | --- |
| `tx_1` | T0 − 14 days − 1 s | just outside the 14-day window |
| `tx_2` | T0 − 14 days | exactly on the 14-day edge → counted |
| `tx_3` | T0 − 1 h − 1 s | outside the 1-hour window, inside 14 days |
| `tx_4` | T0 − 1 h | exactly on the 1-hour edge → counted |
| `tx_5` | T0 − 30 min, other merchant | counts for the card, not for the merchant |
| `tx_6`, `tx_7` | both at T0 | equal times do not count each other |
| `b_now` | T0, card B | amounts 100, 110, 111, 90, 89.99 before it: only 100, 110, 90 are within ±10% of 100.00 |

In [ ]:
fixture_schema = "trans_num string, cc_num string, merchant string, category string, amount decimal(18,2), event_time_str string"

base_rows = [
    ("tx_1_before_14d",  "card_A", "M1", "grocery", Decimal("10.00"),  "2019-01-01 11:59:59"),
    ("tx_2_exact_14d",   "card_A", "M1", "grocery", Decimal("20.00"),  "2019-01-01 12:00:00"),
    ("tx_3_before_1h",   "card_A", "M1", "grocery", Decimal("40.00"),  "2019-01-15 10:59:59"),
    ("tx_4_exact_1h",    "card_A", "M1", "grocery", Decimal("60.00"),  "2019-01-15 11:00:00"),
    ("tx_5_other_merch", "card_A", "M2", "gas",     Decimal("80.00"),  "2019-01-15 11:30:00"),
    ("tx_6_t0_a",        "card_A", "M1", "grocery", Decimal("100.00"), "2019-01-15 12:00:00"),
    ("tx_7_t0_b",        "card_A", "M1", "grocery", Decimal("200.00"), "2019-01-15 12:00:00"),
    # card B: five earlier payments at the same merchant within the last hour
    ("b_1", "card_B", "M1", "grocery", Decimal("100.00"), "2019-01-15 11:50:00"),
    ("b_2", "card_B", "M1", "grocery", Decimal("110.00"), "2019-01-15 11:40:00"),  # upper edge: counted
    ("b_3", "card_B", "M1", "grocery", Decimal("111.00"), "2019-01-15 11:30:00"),  # too high
    ("b_4", "card_B", "M1", "grocery", Decimal("90.00"),  "2019-01-15 11:20:00"),  # lower edge: counted
    ("b_5", "card_B", "M1", "grocery", Decimal("89.99"),  "2019-01-15 11:10:00"),  # too low
    ("b_now", "card_B", "M1", "grocery", Decimal("100.00"), "2019-01-15 12:00:00"),
]


def make_fixture(rows):
    return (
        spark.createDataFrame(rows, schema=fixture_schema)
        .withColumn("event_time", F.col("event_time_str").cast("timestamp_ntz"))
        .drop("event_time_str")
    )


# (card_txn_count_1h, card_txn_count_14d, card_avg_amount_14d, card_merchant_txn_count_1h, similar_amount_count)
expected_features = {
    "tx_1_before_14d":  (0, 0, None,              0, 0),
    "tx_2_exact_14d":   (1, 1, Decimal("10.00"),  1, 0),
    "tx_3_before_1h":   (0, 2, Decimal("15.00"),  0, 0),
    "tx_4_exact_1h":    (1, 3, Decimal("23.33"),  1, 0),
    "tx_5_other_merch": (2, 4, Decimal("32.50"),  0, 0),
    "tx_6_t0_a":        (2, 4, Decimal("50.00"),  1, 0),
    "tx_7_t0_b":        (2, 4, Decimal("50.00"),  1, 0),
    "b_now":            (5, 5, Decimal("100.20"), 5, 3),
}

base_df = make_fixture(base_rows)
base_features = add_history_features(base_df)

actual_features = {
    r["trans_num"]: (
        r["card_txn_count_1h"], r["card_txn_count_14d"], r["card_avg_amount_14d"],
        r["card_merchant_txn_count_1h"], r["card_merchant_similar_amount_count_1h"],
    )
    for r in base_features.collect()
}
for trans_num, expected_values in expected_features.items():
    assert actual_features[trans_num] == expected_values, (
        f"{trans_num}: expected {expected_values}, got {actual_features[trans_num]}"
    )

# Future payments must not change the features of earlier payments.
future_rows = [
    ("future_1m", "card_A", "M1", "grocery", Decimal("999.00"), "2019-01-15 12:01:00"),
    ("future_1d", "card_A", "M1", "grocery", Decimal("500.00"), "2019-01-16 12:00:00"),
]
with_future = add_history_features(make_fixture(base_rows + future_rows))
same_transactions_again = with_future.join(base_df.select("trans_num"), "trans_num").select(base_features.columns)

assert base_features.exceptAll(same_transactions_again).count() == 0, "A future payment changed an earlier feature"
assert same_transactions_again.exceptAll(base_features).count() == 0, "A future payment changed an earlier feature"

print("Self-test passed: window edges, equal times, no history, similar amounts, and no future leakage.")

## Run on real Silver data and save `workspace.sentinel_gold.transaction_features`

In [ ]:
silver = spark.read.table(SILVER_TRANSACTIONS_TABLE)
assert "is_fraud" not in silver.columns, "Fraud labels must not appear in Silver"

silver_rows = silver.count()
assert silver.select("trans_num").distinct().count() == silver_rows, "Silver must have one row per trans_num"

features = add_history_features(silver)

# One feature row per transaction, and "no history" means NULL average.
assert features.count() == silver_rows, "Features changed the number of rows"
assert features.select("trans_num").distinct().count() == silver_rows, "Duplicate trans_num in features"
assert features.filter((F.col("card_txn_count_14d") == 0) & F.col("card_avg_amount_14d").isNotNull()).count() == 0

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FEATURES_TABLE)

print(f"Saved {silver_rows:,} feature rows to {FEATURES_TABLE}.")
display(spark.table(FEATURES_TABLE).orderBy("event_time", "trans_num").limit(25))